<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_15_git_github_system/note_lesson_15_git_github_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 15 — Git + GitHub: перший репозиторій, PR, портфоліо

> Це **не** перше знайомство з Git — воно відбулося на Уроці 1, і відтоді ти вже кілька разів здавав(-ла) домашні роботи через `commit → push → Pull Request`. Досі це було «я знаю, які команди виконувати». Сьогодні — «я розумію, що насправді відбувається», плюс дві практичні навички, яких ще не було: створення **власного** (не fork) репозиторію і мінімальна професійна база для портфоліо.


## Що сьогодні

Після цього заняття ти:

- розумітимеш різницю між working tree, staging area, local repository і remote — не як абстрактні слова, а як реальні місця, де побували твої файли;
- зможеш пояснити, що таке commit graph і навіщо потрібні branches;
- створиш власний (не fork) репозиторій на GitHub;
- навмисно створиш і розв'яжеш **merge conflict** — побачиш його один раз у контрольованих умовах, а не вперше в паніці посеред домашньої роботи;
- приведеш один зі своїх попередніх проєктів курсу до мінімального «портфоліо-вигляду»: README, `.gitignore`, чиста історія комітів.


## RETRIEVE

Без підглядання в конспект чи документацію — коротко дай відповідь:

1. Які команди ти виконуєш між «я написав(-ла) код домашнього завдання» і «PR відкритий і чекає на перевірку»? Перерахуй по порядку.
2. Що показує `git status`?
3. Чим `origin` відрізняється від `upstream`?

Якщо на якесь із питань відповіді нема — не страшно, це і є привід для сьогоднішнього заняття. Якщо є — сьогодні дізнаєшся, **чому** ці команди працюють саме так, а не тільки що вони роблять.


## CONCEPT — три місця, де живуть твої файли

Досі `git add` і `git commit` могли виглядати як один ритуал «зберегти зміни». Насправді це два окремих кроки між трьома різними місцями:

```text
Working Tree                Staging Area                  Local Repository
(файли на диску,     --git add-->  (список: «ці зміни    --git commit-->  (постійний запис
 як у будь-якій                     підуть у наступний                      в історії Git)
 папці)                             commit»)
```

**Working tree** — це просто файли на диску, які ти редагуєш у PyCharm. Git постійно порівнює їх зі своєю останньою версією і показує різницю через `git status`.

**Staging area** (її ще звуть *index*) — це не копія файлів, а **список намірів**: «ось ці конкретні зміни підуть у наступний commit». Навіщо цей проміжний крок узагалі потрібен, якщо можна було б одразу `git commit` усе, що змінилось?

Уяви: ти виправив(-ла) баг **і** попутно поекспериментував(-ла) з іншим, ще не готовим шматком коду в тому самому файлі. `git add -p` (або обережний вибір файлів) дозволяє закомітити **тільки виправлення бага** одним чистим commit, залишивши експеримент незакомміченим. Без staging area довелося б комітити все відразу або нічого.

**Local repository** — постійна історія на твоєму комп'ютері: кожен `git commit` дописує в неї новий, незмінний запис. Саме сюди `git log` заглядає, коли показує історію.

`git status` — це фактично питання «що зараз у working tree, а що вже в staging area, і чим це відрізняється від того, що в local repository?»:

```text
On branch homework-01

Changes not staged for commit:      ← working tree відрізняється від repository, але git add ще не робили
  modified: task1.py

Changes to be committed:            ← вже в staging area, чекає на git commit
  modified: task2.py

Untracked files:                    ← Git ще взагалі не знає про цей файл
  new_file.py
```


## CONCEPT — commit graph

Кожен commit не існує сам по собі — він **вказує на попередній commit** (свого «батька»). Тому `git log` показує не випадковий список, а ланцюжок:

```mermaid
gitGraph
   commit id: "init"
   commit id: "lesson 01"
   commit id: "lesson 02"
   commit id: "lesson 03"
```

Це і є **commit graph** — послідовність знімків проєкту, де кожен наступний «пам'ятає», з якого попереднього він виріс. Коли гілок стає кілька, це вже не пряма лінія, а справжній граф (побачиш нижче, у розділі про branches).

Кожен commit фіксує: автора, дату, повідомлення і **весь стан проєкту** на той момент (технічно — не копію кожного файлу заново, а посилання на ті файли, які реально змінились; решта переіспользуються). Тому `git checkout` на старий commit миттєво повертає весь проєкт у той стан, яким він був тоді.


## CONCEPT — branch — це не копія файлів, а рухомий вказівник

Найчастіша інтуїтивна помилка: думати, що `git checkout -b homework-05` копіює всі файли в нове місце. Насправді branch — це просто **іменований вказівник на конкретний commit**, який рухається вперед разом із новими commit-ами в цій гілці:

```mermaid
gitGraph
   commit id: "lesson"
   branch homework-05
   commit id: "task1"
```

Саме тому створення нової гілки — миттєва, «дешева» операція: Git не копіює жодного файлу, він просто ставить новий підписаний прапорець на той самий commit, де ти зараз стоїш. `git switch` (або `git checkout branch_name`) — це не «завантажити копію», а «перемкнути working tree на стан, куди вказує цей прапорець».

`*` у виводі `git branch` показує, на якому прапорці ти стоїш зараз:

```text
* main
homework-01
homework-02
```


## CONCEPT — `origin` і `upstream` — це просто імена-закладки для URL

`origin` та `upstream` не мають нічого магічного — це звичайні **іменовані посилання** на віддалені репозиторії, збережені в конфігурації твого локального репозиторію. Перевірити:

```bash
git remote -v
```

```text
origin   → https://github.com/ТВІЙ_НІК/PY-Course-....git   (твій fork)
upstream → https://github.com/NikoriakViktot/PY-Course-....git   (репозиторій викладача)
```

| remote | на що вказує | коли використовуєш |
|---|---|---|
| `origin` | твій fork | `git push origin homework-01` — здати свою роботу |
| `upstream` | репозиторій викладача | `git pull upstream main` — забрати нові матеріали курсу |

Ці імена — просто зручні ярлики (`origin` і `upstream` — конвенція, не вимога Git); технічно можна дати їм будь-яку назву. Докладніше про підключення `upstream` — [Fork і Clone](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/00_getting_started/github/fork_and_clone/), крок 3️⃣.


## CONCEPT — push і pull: синхронізація з GitHub

```mermaid
flowchart LR
A[Local Repository] -- git push --> B[GitHub Repository]
B -- git pull --> A
```

`git push origin homework-01` — відправляє commit-и з твоєї локальної гілки на відповідну гілку на GitHub. Якщо гілка на GitHub ще не існує — `-u` (`git push -u origin homework-01`) створює її і одразу запам'ятовує зв'язок, щоб надалі вистачало просто `git push`.

`git pull` — насправді два кроки одразу:

```text
git fetch   → завантажує нові commit-и з remote, але НЕ чіпає твій working tree
git merge   → об'єднує щойно завантажені commit-и з твоєю поточною гілкою
```

Це пояснює, чому іноді `git pull` завершується конфліктом (детальніше — нижче): `fetch` завжди проходить чисто, а от `merge` — ні, якщо той самий рядок змінили з обох боків.


## CONCEPT — merge: об'єднання гілок

`git merge branch_name`, стоячи на `main`, приносить commit-и з `branch_name` у `main`:

```mermaid
gitGraph
   commit
   commit
   branch homework
   commit
   commit
   checkout main
   merge homework
```

Є два випадки:

- **Fast-forward** — поки ти працював(-ла) у `homework`, `main` взагалі не змінювався. Git просто пересуває прапорець `main` вперед, до останнього commit-а `homework`. Жодного нового commit-а для цього не створюється.
- **Справжній merge (three-way merge)** — обидві гілки отримали нові commit-и незалежно одна від одної. Git створює новий, спеціальний **merge commit** із двома батьками одразу — саме тут можливий конфлікт.


## CONCEPT — merge conflict: коли Git не може вирішити сам

Git вміє автоматично об'єднувати зміни, якщо вони торкаються **різних** рядків файлу. Конфлікт виникає тільки тоді, коли дві гілки змінили **той самий рядок** по-різному — Git фізично не може вгадати, яка версія правильна, і зупиняється, щоб запитати тебе.

Коли це стається, Git залишає у файлі спеціальні маркери прямо на місці конфлікту:

```text
<<<<<<< HEAD
змінено з branch-a
=======
змінено з branch-b
>>>>>>> branch-b
```

Читається так:
- усе між `<<<<<<< HEAD` і `=======` — версія з гілки, на якій ти зараз стоїш (`HEAD`);
- усе між `=======` і `>>>>>>> branch-b` — версія з гілки, яку вливаєш (`branch-b`).

**Розв'язання конфлікту — завжди одні й ті самі 4 кроки:**

1. Відкрий файл, вирішіть, яка версія (або яка комбінація обох) має залишитись.
2. Видали самі маркери (`<<<<<<<`, `=======`, `>>>>>>>`) — вони не частина коду, Git залишив їх лише як підказку.
3. `git add ІМ'Я_ФАЙЛУ` — позначає конфлікт розв'язаним.
4. `git commit` — Git сам запропонує готове повідомлення на кшталт `Merge branch 'branch-b'`, досить просто підтвердити.

Якщо на середині стало незрозуміло і хочеться почати заново — `git merge --abort` повністю скасовує merge і повертає все точно в той стан, який був до нього. Це безпечна кнопка «відміни», нею не соромно користуватись.

Практично ти зробиш це руками в наступному розділі — на маленькому, навмисно підготовленому прикладі.


## CREATE

Дві практичні вправи. Обидві виконуються в терміналі (PowerShell / термінал PyCharm), не в цьому ноутбуці — Git тут не про Python-код, а про роботу з файлами й GitHub. Команди позначені блоками **🖐 Виконай у терміналі**.


### 🖐 Крок 1 — створи власний (не fork) репозиторій

Це відрізняється від того, що ти робив(-ла) на Уроці 1: там ти форкав репозиторій курсу. Тепер створюєш **свій власний**, порожній, з нуля.

Повна інструкція з двома сценаріями (починаєш із GitHub / код уже є локально) — [Як створити свій репозиторій](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/00_getting_started/github/create_repository/). Коротко (сценарій А — код ще не писав(-ла)):

```bash
# на github.com/new: назва, Public, галочка "Add a README", .gitignore = Python
# потім клонуй його собі, як робив(-ла) з fork на Уроці 1

cd шлях/куди/склонував(-ла)
echo "print('hello from my own repo')" > main.py
git add main.py
git commit -m "Add main.py"
git push
```

Після цього в тебе є перший **справді власний** репозиторій на GitHub — основа для капстоуна (Урок 17) чи будь-якого пет-проєкту.


### 🖐 Крок 2 — навмисно створи merge conflict

Зроби це в окремій, тестовій папці (не у своєму курсовому fork і не в щойно створеному репозиторії з кроку 1) — вона потрібна лише для цієї вправи, і її можна видалити після.

```bash
mkdir conflict_demo && cd conflict_demo
git init

echo "рядок 1" > notes.txt
git add notes.txt
git commit -m "Initial notes"

# перша гілка змінює той самий рядок
git checkout -b branch-a
echo "змінено з branch-a" > notes.txt
git commit -am "Edit from branch-a"

# друга гілка — від main, змінює той самий рядок по-іншому
git checkout main
git checkout -b branch-b
echo "змінено з branch-b" > notes.txt
git commit -am "Edit from branch-b"
```

Тепер об'єднай обидві в `main`:

```bash
git checkout main
git merge branch-a
```

Це пройде **без конфлікту** — `main` після `git init` ще не рухався, тож це fast-forward (`main` просто «наздоганяє» `branch-a`).

```bash
git merge branch-b
```

**А ось тут — конфлікт.** `main` (уже = `branch-a`) і `branch-b` незалежно змінили той самий рядок `notes.txt` по-різному — Git не може вирішити сам і зупиняється.


### 🖐 Крок 3 — розв'яжи конфлікт

Відкрий `notes.txt` — усередині побачиш ті самі маркери, що в CONCEPT вище:

```text
<<<<<<< HEAD
змінено з branch-a
=======
змінено з branch-b
>>>>>>> branch-b
```

Виріши, яка версія залишається (або залиш обидва рядки, якщо це має сенс), видали маркери, і заверши merge:

```bash
# відредагуй notes.txt вручну — прибери маркери, залиш потрібний текст
git add notes.txt
git commit
```

Перевір результат:

```bash
git log --oneline --graph
```

Побачиш merge commit із **двома** батьками — саме той випадок "справжнього merge", а не fast-forward, про який йшлося в CONCEPT.

Якщо на якомусь кроці стало незрозуміло — `git merge --abort` поверне все назад, і можна спробувати ще раз.


## TRANSFER — мінімальне портфоліо

Обери один зі своїх попередніх проєктів курсу (наприклад, один із виконаних домашніх завдань або власний репозиторій із кроку 1) і приведи його до мінімального «портфоліо-вигляду». «Портфоліо» тут — **не** особистий брендинг, а мінімальна професійна база: те, що будь-який рецензент чи роботодавець очікує побачити, відкривши репозиторій вперше.

### Чеклист README

Хороший README відповідає на три питання за 10 секунд читання:

1. **Що це** — заголовок і одне речення опису.
2. **Як це запустити** — команди встановлення залежностей і запуску (блоком коду).
3. (за бажанням) **Як це виглядає** — скріншот або приклад виводу.

### Чеклист `.gitignore`

Мінімум для Python-проєкту: `__pycache__/`, `.venv/` (чи `venv/`), `*.pyc`, `.env` (якщо є секрети/ключі). Готовий повний шаблон — той самий, що пропонує GitHub при створенні репозиторію (сценарій А в [Як створити свій репозиторій](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/00_getting_started/github/create_repository/)).

### Чеклист історії комітів

| Погано | Добре |
|---|---|
| `fix` | `Fix off-by-one error in page counter` |
| `asdf` | `Add input validation for negative amounts` |
| `final version 2  FINAL` | `Add README with setup instructions` |
| один величезний commit «весь проєкт» | кілька осмислених commit-ів, кожен — одна логічна зміна |

Правило: повідомлення в наказовому способі (`Add`, `Fix`, `Remove`, не `Added`/`Fixed`) і достатньо конкретне, щоб через місяць за одним рядком `git log --oneline` зрозуміти, що саме змінилось.

Нижче — невеликий Python-інструмент, який перевіряє README і `.gitignore` за цими чеклистами. Спочатку подивись, як він працює на прикладах, потім став `README_PATH`/`GITIGNORE_PATH` на свій реальний проєкт і запусти ще раз.


In [1]:
def check_readme(text: str) -> dict[str, bool]:
    """Мінімальний чеклист README: заголовок, опис, інструкція запуску."""
    has_title = text.strip().startswith("#")
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip() and not p.strip().startswith("#")]
    has_description = len(paragraphs) > 0
    lowered = text.lower()
    has_run_instructions = ("```" in text) or ("run" in lowered) or ("запуст" in lowered) or ("встанов" in lowered)
    return {
        "має заголовок (# Назва)": has_title,
        "має опис проєкту": has_description,
        "пояснює, як запустити": has_run_instructions,
    }


good_readme = """# Weather CLI

Консольна утиліта, яка показує поточну погоду для введеного міста.

## Встановлення

```bash
pip install -r requirements.txt
```

## Запуск

```bash
python weather_cli.py Kyiv
```
"""

bad_readme = "TODO: написати опис"

for name, text in [("good_readme", good_readme), ("bad_readme", bad_readme)]:
    print(name, "->", check_readme(text))

assert all(check_readme(good_readme).values())
assert not all(check_readme(bad_readme).values())
print("Чеклист README перевірено на прикладах.")

good_readme -> {'має заголовок (# Назва)': True, 'має опис проєкту': True, 'пояснює, як запустити': True}
bad_readme -> {'має заголовок (# Назва)': False, 'має опис проєкту': True, 'пояснює, як запустити': False}
Чеклист README перевірено на прикладах.


In [2]:
def check_gitignore(text: str, required=("__pycache__/", ".venv", "*.pyc", ".env")) -> dict[str, bool]:
    """Мінімальний чеклист .gitignore для Python-проєкту."""
    lines = [line.strip() for line in text.splitlines()]
    return {pattern: any(pattern in line for line in lines) for pattern in required}


good_gitignore = "__pycache__/\n*.pyc\n.venv/\n.env\n"
bad_gitignore = "*.log\n"

for name, text in [("good_gitignore", good_gitignore), ("bad_gitignore", bad_gitignore)]:
    print(name, "->", check_gitignore(text))

assert all(check_gitignore(good_gitignore).values())
assert not all(check_gitignore(bad_gitignore).values())
print("Чеклист .gitignore перевірено на прикладах.")

good_gitignore -> {'__pycache__/': True, '.venv': True, '*.pyc': True, '.env': True}
bad_gitignore -> {'__pycache__/': False, '.venv': False, '*.pyc': False, '.env': False}
Чеклист .gitignore перевірено на прикладах.


### 🖐 Застосуй до свого проєкту

```python
# BEGIN SOLUTION
README_PATH = "шлях/до/твого/README.md"
GITIGNORE_PATH = "шлях/до/твого/.gitignore"

with open(README_PATH, encoding="utf-8") as f:
    print(check_readme(f.read()))

with open(GITIGNORE_PATH, encoding="utf-8") as f:
    print(check_gitignore(f.read()))
# END SOLUTION
```

Заміни шляхи на реальні файли свого проєкту (з кроку 1 або будь-якого попереднього домашнього завдання) і запусти в своєму середовищі. Мета — щоб усі значення в обох словниках стали `True`. Якщо ні — допиши README чи `.gitignore` за чеклистом вище, збережи, запусти перевірку знову.

Це не виконується автоматично в цьому ноутбуці (шлях `"шлях/до/твого/README.md"` навмисно не існує) — це вправа, яку ти доводиш до `True` у власному середовищі, на власному файлі.


## Самоперевірка

Коротко, без підглядання:

1. У чому різниця між working tree, staging area і local repository? Яку команду ти виконуєш, щоб перейти з одного в інше?
2. Що таке branch технічно — окрема копія файлів чи щось інше?
3. `origin` і `upstream` — чим відрізняються і звідки Git знає, куди вони вказують?
4. Побач цей фрагмент і поясни, що станеться далі:
   ```text
   <<<<<<< HEAD
   version A
   =======
   version B
   >>>>>>> feature-branch
   ```
5. Назви три речі, які обов'язково має мати репозиторій, щоб виглядати «портфоліо-готовим».

*(Відповіді не здаються окремо — перевір себе усно чи письмово перед тим, як рухатись далі.)*


## Далі

Урок 16 (П3. Хеш-структури) продовжує алгоритмічну лінію курсу. Git/GitHub тепер — не окрема тема, а фоновий інструмент: кожне наступне домашнє завдання і особливо капстоун-проєкт на Уроці 17 будуть використовувати саме ці навички — власний репозиторій, чисту історію комітів і (сподіваємось, рідше) впевнене розв'язання конфліктів.

👉 Практичні інструкції залишаються тут, під рукою, коли знадобляться: [SSH-ключі](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/00_getting_started/github/ssh_keys/) · [Fork і Clone](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/00_getting_started/github/fork_and_clone/) · [Pull Request](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/00_getting_started/github/pull_request/) · [Як створити свій репозиторій](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/00_getting_started/github/create_repository/) · [Git шпаргалка](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/git-cheatsheet/).
